In [ ]:
from openai import OpenAI
import pandas as pd
import os
import json
import re
import time

# -----------------------------
# LLM Client Setup
# -----------------------------
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")
model_name = "google/gemma-3-1b"

INPUT_FILE = "filtered_tweets_2014_Q4.csv.gz"
OUTPUT_FILE = "filtered_tweets_2014_Q4_classified.csv.gz"

# -----------------------------
# JSON Schema for classification
# -----------------------------
classification_schema = {
    "type": "json_schema",
    "json_schema": {
        "name": "AgeismClassification",
        "schema": {
            "type": "object",
            "properties": {
                "classification": {
                    "type": "string",
                    "enum": ["Not relevant", "Ageist", "Age-positive"]
                }
            },
            "required": ["classification"]
        },
    }
}

# -----------------------------
# Load input
# -----------------------------
df = pd.read_csv(INPUT_FILE, compression="gzip", dtype=str)
df["classification"] = ""

# -----------------------------
# Loop through tweets
# -----------------------------
for i, row in df.iterrows():
    start_time = time.time()
    text = row["text"]  # make sure your tweet text column is called "text"

    prompt_text = f"""
    You are an assistant analyzing social media posts for age-related content.  
    Your task is to classify each post into one of three categories:

    1. "Not relevant" – if the post is not related to age, aging, or ageism.  

    2. "Ageist" – if the post contains any negative, harmful, or stereotypical language about aging or older adults.  
       Examples and keywords include (but are not limited to):  
       alone, loneliness, isolated, diseased, decrepit, infirm, confused, forgetful, memory loss, misplaces, dying, waiting to die, retire already, time to go, dementia, senile, senility, incompetent, dependent, decline, alzheimer's, deteriorated, shaky, wrinkled, grumpy, cranky, given up, walks slowly, slow, helpless, aches and pains, frail, fragile, weak, vulnerable, not tech-savvy, technophobe, technologically challenged, outdated, stuck in the past, old-fashioned, out-of-touch, can't learn, can't change, despondent, impaired, shrew, curmudgeon, complaining, ill-tempered, recluse, timid, naïve, nosy neighbor, put someone out to pasture, losing it, bag lady, rigid, feeble, irrelevant, doddering, fossil, geezer, fuddy-duddy, coot, old fart, old hag, has-been, advanced age, old as Methuselah, antediluvian.  

    3. "Age-positive" – if the post highlights positive qualities of aging or older adults.  
       Examples and keywords include (but are not limited to):  
       astute, insightful, enlightened, creative, advise, improving, learned, accomplished, sage, alert, guidance, wise, wisdom, wiser, family-oriented, capable, active, positive outlook, well-groomed, will-to-live, full of life, more freedom and time for new interests, continue to grow as a person, appreciate things more, nurturing, warm, caring, kind, loving, respected, honored, role model, mentor, family anchor, legacy, heritage, golden years, aging gracefully, graceful aging, patriotic, nostalgic, embrace aging, age positivity, experienced, venerable, mature, spry, zesty, feisty, spirited.  

    Return only a single category label:  
    "Not relevant", "Ageist", or "Age-positive".

    Post: {text}
    """

    # LLM call
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": [{"type": "text", "text": prompt_text.strip()}]}],
        response_format=classification_schema,
    )

    raw = response.choices[0].message.content.strip()
    #print(f"🏷️ LLM output raw: {raw}")

    # Parse JSON safely
    try:
        parsed = json.loads(raw)
        classification = parsed.get("classification", "Not relevant")
    except Exception:
        classification = "Not relevant"

    df.at[i, "classification"] = classification

    print(f"\n📄 Tweet {i+1} of {len(df)}")
    print(f"🏷️ Classification: {classification}")
    #print(f"⏱️ Runtime: {round(time.time() - start_time, 2)} seconds")

# -----------------------------
# Save output
# -----------------------------
df.to_csv(OUTPUT_FILE, index=False, compression="gzip")
print(f"\n✅ Done. Saved to {OUTPUT_FILE}")
